# Register and handle tools using a dictionary
This example used local models, but can easily be updated to use frontier models 

In [ ]:
import json
from openai import OpenAI
import gradio as gr
import inspect

In [ ]:
# Initialization
    
MODEL = "gpt-oss:20b"
ollama = OpenAI(api_key="ollama", base_url="http://localhost:11434/v1")

tools = {}

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [ ]:
def register_tool(func:callable, description:str):
    """Registers a function as a tool for the LLM to use.

    Parameter types are derived from function annotations, and descriptions are derived from the function docstrings
    Args:
        func (callable): The function to register as a tool.
        description (str): A brief description of what the tool does.
    """
    parameters = {}
    sig = inspect.signature(func)
    for param_name, param in sig.parameters.items():
        parameters[param_name] = {"type": param.annotation.__name__ if param.annotation != inspect.Parameter.empty else "str", "description": get_description(func.__doc__, param_name)}
    tools[func.__name__] = {"function": func,
                             "json": {
                                "name": func.__name__,
                    "description": description,
                    "parameters": {
                        "type": "object",
                        "properties": parameters,
                    },
                    "required": [
                        name for name, param in sig.parameters.items()
                        if param.default is inspect.Parameter.empty 
                        and param.kind not in (inspect.Parameter.VAR_POSITIONAL, inspect.Parameter.VAR_KEYWORD)
                    ],
                    "additional_properties": False,
                    }
                }

def get_description(docstring, param_name):
    """A naive way to extract parameter descriptions from a docstring."""
    if not docstring:
        return ""
    lines = docstring.splitlines()
    for line in lines:
        line = line.strip()
        if param_name in line and ':' in line:
            return line.partition(':')[2].strip()
    return ""

In [ ]:
def handle_tool_calls(message):
    """Handles tool calls from the model by executing the corresponding functions and returning their results."""
    responses = []
    for tool_call in message.tool_calls:
        print(f"Got tool call {tool_call}")
        func = tools.get(tool_call.function.name, {})
        if func:
            arguments = json.loads(tool_call.function.arguments)
            print(f"Calling function {func['function']} with arguments {arguments}")
            result = func['function'](**arguments)
            print(f"Got result {result} from tool call")
            responses.append({
                "role": "tool",
                "content": result or "",
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
def chat(message, history):
    tools_for_model = [{"type": "function", "function": tools[t]["json"]} for t in tools]
    
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools_for_model)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools_for_model)
    
    return response.choices[0].message.content

In [ ]:
import sqlite3
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
def get_ticket_price(city:str) -> str:
    """
    Get the price of a ticket to a specified city.
    Args:
        destination_city (str): The name of the destination city.
    Returns:
    """
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
def set_ticket_price(city:str, price:float):
    """Set the price of a ticket to a specified city.
    Args:
        city (str): The name of the city.
        price (float): The price of the ticket.
    Returns:
        None
    """
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
register_tool(get_ticket_price, "Get the price of a ticket to a specified city.")
register_tool(set_ticket_price, "Set the price of a ticket for a specified city.")

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()